# 04 Virtual Directed Evolution

This notebook simulates Round 0 historical experiments followed by three virtual recommendation rounds. It compares random selection, direct fitness-model ranking, a one-shot LLM Agent recommendation list, a feedback-driven local Critic strategy, a true per-round OpenAI LLM Critic strategy, and a knowledge-enhanced LLM Agent strategy.

When `RUN_OPENAI_PER_ROUND = True` and `OPENAI_API_KEY` is available, `openai_feedback_llm_agent` calls OpenAI once per virtual round. Each call receives previous true-fitness feedback, critiques success/failure patterns, and selects the next Top-k candidates from a supplied candidate pool.


In [ ]:
from pathlib import Path
import os

import pandas as pd
from dotenv import load_dotenv

try:
    from openai import OpenAI
except ImportError:
    OpenAI = None

from knowledge_base import apply_mutation_count_constraint, score_mutation_rules, weighted_knowledge_score
from llm_agent_core import add_mutation_features, build_designer_pool, load_two_vs_many, summarize_history
from report_figures import summarize_rounds
from virtual_evolution import (
    score_candidates_with_feedback,
    select_openai_feedback_candidates,
    summarize_feedback,
    select_model_top_candidates,
    select_random_candidates,
    simulate_iterative_evolution,
    summarize_recommended_positions,
)

load_dotenv()
ARTIFACTS = Path("artifacts")
ARTIFACTS.mkdir(exist_ok=True)
ROUNDS = 3
TOP_K = 10
MAX_MUTATIONS_FOR_KNOWLEDGE = 4
RUN_OPENAI_PER_ROUND = bool(os.getenv("OPENAI_API_KEY"))
LLM_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.6-luna")
OPENAI_CANDIDATE_POOL_SIZE = 80
openai_client = OpenAI() if RUN_OPENAI_PER_ROUND and OpenAI is not None else None

print("OpenAI per-round enabled:", openai_client is not None)


In [ ]:
observed_df, candidate_pool = load_two_vs_many("two_vs_many.csv")
observed_features = add_mutation_features(observed_df)

test_truth = pd.read_csv("test.csv").reset_index(drop=True)
test_truth["candidate_id"] = [f"C{i:05d}" for i in range(len(test_truth))]
test_candidates = add_mutation_features(test_truth)

top_history, mutation_summary = summarize_history(observed_df)
designer_pool = build_designer_pool(candidate_pool, mutation_summary, pool_size=5000)
truth_pool = designer_pool.merge(
    test_candidates[["candidate_id", "target"]],
    on="candidate_id",
    how="left",
)

prediction_path = ARTIFACTS / "test_esm2_mlp_predictions.csv"
if prediction_path.exists():
    predictions = pd.read_csv(prediction_path).reset_index(drop=True)
    predictions["candidate_id"] = [f"C{i:05d}" for i in range(len(predictions))]
    predictions = predictions.rename(columns={"predicted_fitness": "esm_predicted_fitness"})
    truth_pool = truth_pool.merge(
        predictions[["candidate_id", "esm_predicted_fitness"]],
        on="candidate_id",
        how="left",
    )
else:
    truth_pool["esm_predicted_fitness"] = truth_pool["historical_prior"]

truth_pool["predicted_fitness"] = truth_pool["esm_predicted_fitness"].fillna(truth_pool["historical_prior"])
truth_pool["prediction_error"] = truth_pool["predicted_fitness"] - truth_pool["target"]

rule_scores = truth_pool["mutations"].apply(score_mutation_rules).apply(pd.Series)
truth_pool = pd.concat([truth_pool.reset_index(drop=True), rule_scores.reset_index(drop=True)], axis=1)
truth_pool = weighted_knowledge_score(truth_pool, historical_weight=0.7, rule_weight=0.3)

print("Initial observed training variants:", len(observed_features))
print("Hidden test candidates for virtual evaluation:", len(truth_pool))
display(top_history[["mutated_region", "target", "mutations", "num_mutations"]].head(10))
display(truth_pool[[
    "candidate_id",
    "mutations",
    "num_mutations",
    "target",
    "predicted_fitness",
    "prediction_error",
    "historical_prior",
    "rule_score",
    "knowledge_enhanced_score",
]].head())


In [ ]:
agent_path = ARTIFACTS / "llm_agent_final_recommendations.csv"
agent_seed_ids = []
if agent_path.exists():
    agent_seed_ids = pd.read_csv(agent_path)["candidate_id"].astype(str).tolist()
agent_seed_ids = list(dict.fromkeys(agent_seed_ids))
feedback_records = []
openai_feedback_records = []


def score_with_current_history(current_observed, available_candidates):
    _, current_mutation_summary = summarize_history(current_observed, top_history_count=200)
    scored = build_designer_pool(
        available_candidates[["sequence", "candidate_id"]],
        current_mutation_summary,
        pool_size=len(available_candidates),
    )
    scored = scored.drop(columns=["target", "predicted_fitness", "prediction_error"], errors="ignore")
    keep_cols = [
        "candidate_id",
        "target",
        "esm_predicted_fitness",
        "predicted_fitness",
        "prediction_error",
        "knowledge_enhanced_score",
    ]
    return scored.merge(available_candidates[keep_cols], on="candidate_id", how="left")


def random_strategy(current_observed, available_candidates, round_index, top_k):
    return select_random_candidates(available_candidates["candidate_id"], top_k=top_k, seed=round_index)


def model_strategy(current_observed, available_candidates, round_index, top_k):
    return select_model_top_candidates(available_candidates, top_k=top_k, score_col="predicted_fitness")


def feedback_driven_llm_agent_strategy(current_observed, available_candidates, round_index, top_k):
    dynamic_pool = score_with_current_history(current_observed, available_candidates)
    feedback = summarize_feedback(current_observed, fitness_col="target")
    feedback_records.append({
        "strategy": "feedback_driven_llm_agent",
        "round": round_index,
        "observed_count": len(current_observed),
        "successful_positions": feedback["successful_positions"],
        "failed_positions": feedback["failed_positions"],
        "successful_mutations": feedback["successful_mutations"][:20],
        "failed_mutations": feedback["failed_mutations"][:20],
        "fitness_threshold_high": feedback["fitness_threshold_high"],
        "fitness_threshold_low": feedback["fitness_threshold_low"],
    })
    scored = score_candidates_with_feedback(
        dynamic_pool,
        feedback,
        max_mutations=MAX_MUTATIONS_FOR_KNOWLEDGE,
        predicted_col="predicted_fitness",
        knowledge_col="knowledge_enhanced_score",
    )
    constrained = scored[scored["num_mutations"] <= MAX_MUTATIONS_FOR_KNOWLEDGE].copy()
    if constrained.empty:
        constrained = scored
    return (
        constrained.sort_values(["feedback_agent_score", "predicted_fitness"], ascending=False)
        .head(top_k)["candidate_id"]
        .astype(str)
        .tolist()
    )


def openai_feedback_llm_agent_strategy(current_observed, available_candidates, round_index, top_k):
    dynamic_pool = score_with_current_history(current_observed, available_candidates)
    if openai_client is None:
        return feedback_driven_llm_agent_strategy(current_observed, available_candidates, round_index, top_k)

    selected_ids, decision, ranked_pool = select_openai_feedback_candidates(
        openai_client,
        LLM_MODEL,
        current_observed,
        dynamic_pool,
        round_index=round_index,
        top_k=top_k,
        candidate_pool_size=OPENAI_CANDIDATE_POOL_SIZE,
        max_mutations=MAX_MUTATIONS_FOR_KNOWLEDGE,
        predicted_col="predicted_fitness",
        knowledge_col="knowledge_enhanced_score",
        fitness_col="target",
    )
    openai_feedback_records.append({
        "round": round_index,
        "observed_count": len(current_observed),
        "selected_candidate_ids": selected_ids,
        "critic_feedback": decision.critic_feedback,
        "successful_patterns": decision.successful_patterns,
        "failed_patterns": decision.failed_patterns,
        "next_round_strategy": decision.next_round_strategy,
        "limitations": decision.limitations,
    })
    return selected_ids


def llm_agent_strategy(current_observed, available_candidates, round_index, top_k):
    # Legacy LLM-agent seed strategy: uses the one-shot LLM recommendation list if available,
    # then falls back to history-based ranking. Kept as an ablation.
    dynamic_pool = score_with_current_history(current_observed, available_candidates)
    seeded = dynamic_pool[dynamic_pool["candidate_id"].isin(agent_seed_ids)].copy()
    seeded_ids = select_model_top_candidates(seeded, top_k=min(top_k, len(seeded)), score_col="historical_prior") if not seeded.empty else []
    if len(seeded_ids) >= top_k:
        return seeded_ids[:top_k]
    fallback_pool = dynamic_pool[~dynamic_pool["candidate_id"].isin(seeded_ids)]
    fallback_ids = select_model_top_candidates(fallback_pool, top_k=top_k - len(seeded_ids), score_col="historical_prior")
    return seeded_ids + fallback_ids


def knowledge_enhanced_strategy(current_observed, available_candidates, round_index, top_k):
    dynamic_pool = score_with_current_history(current_observed, available_candidates)
    constrained = apply_mutation_count_constraint(dynamic_pool, max_mutations=MAX_MUTATIONS_FOR_KNOWLEDGE)
    if constrained.empty:
        constrained = dynamic_pool
    rule_scores = constrained["mutations"].apply(score_mutation_rules).apply(pd.Series)
    constrained = pd.concat([constrained.reset_index(drop=True), rule_scores.reset_index(drop=True)], axis=1)
    constrained = weighted_knowledge_score(constrained, historical_weight=0.7, rule_weight=0.3)
    return select_model_top_candidates(constrained, top_k=top_k, score_col="knowledge_enhanced_score")


strategies = {
    "random_mutation": random_strategy,
    "fitness_model_direct": model_strategy,
    "llm_agent_one_shot": llm_agent_strategy,
    "feedback_driven_llm_agent": feedback_driven_llm_agent_strategy,
    "openai_feedback_llm_agent": openai_feedback_llm_agent_strategy,
    "knowledge_enhanced_llm_agent": knowledge_enhanced_strategy,
}

results = simulate_iterative_evolution(
    observed_features,
    truth_pool,
    strategies,
    rounds=ROUNDS,
    top_k=TOP_K,
    evaluator_col="target",
)
results.to_csv(ARTIFACTS / "virtual_evolution_results.csv", index=False)

recommended_topk = results[results["round"] > 0].copy()
recommended_topk.to_csv(ARTIFACTS / "virtual_evolution_topk_by_round.csv", index=False)

feedback_df = pd.DataFrame(feedback_records)
feedback_df.to_csv(ARTIFACTS / "virtual_evolution_feedback_by_round.csv", index=False)

openai_feedback_df = pd.DataFrame(openai_feedback_records)
openai_feedback_df.to_csv(ARTIFACTS / "openai_virtual_evolution_feedback.csv", index=False)
openai_recommendations = recommended_topk[recommended_topk["strategy"] == "openai_feedback_llm_agent"].copy()
openai_recommendations.to_csv(ARTIFACTS / "openai_virtual_evolution_recommendations.csv", index=False)

print("Local feedback records generated:", len(feedback_df))
print("OpenAI feedback records generated:", len(openai_feedback_df))
if not openai_feedback_df.empty:
    display(openai_feedback_df[[
        "round",
        "selected_candidate_ids",
        "critic_feedback",
        "successful_patterns",
        "failed_patterns",
        "next_round_strategy",
        "limitations",
    ]])
else:
    print("OpenAI per-round LLM was not enabled; set OPENAI_API_KEY in .env and rerun this notebook.")

display(recommended_topk[[
    "strategy",
    "round",
    "selection_rank",
    "candidate_id",
    "mutations",
    "num_mutations",
    "predicted_fitness",
    "fitness",
    "prediction_error",
    "round_best_fitness",
    "observed_count_before_round",
]])


In [ ]:
summary = summarize_rounds(recommended_topk)
summary["mean_fitness_change_vs_previous_round"] = summary.groupby("strategy")["mean_fitness"].diff()
summary["best_fitness_change_vs_previous_round"] = summary.groupby("strategy")["best_fitness"].diff()
summary["mean_fitness_improved_vs_previous_round"] = summary["mean_fitness_change_vs_previous_round"].fillna(0) > 0
summary["best_fitness_improved_vs_previous_round"] = summary["best_fitness_change_vs_previous_round"].fillna(0) > 0

summary.to_csv(ARTIFACTS / "virtual_evolution_summary.csv", index=False)
display(summary)

position_summary = summarize_recommended_positions(recommended_topk)
position_summary.to_csv(ARTIFACTS / "virtual_evolution_key_position_summary.csv", index=False)
top_positions = (
    position_summary
    .sort_values(["strategy", "round", "mutation_frequency", "mutation_count"], ascending=[True, True, False, False])
    .groupby(["strategy", "round"])
    .head(8)
    .reset_index(drop=True)
)
display(top_positions)

ax = summary.pivot(index="round", columns="strategy", values="best_fitness").plot(marker="o", figsize=(8, 4))
ax.set_ylabel("Best true fitness in recommended Top-k")
ax.set_title("Best recommendation fitness over 3 virtual rounds")
fig = ax.get_figure()
fig.tight_layout()
fig.savefig(ARTIFACTS / "virtual_evolution_curve.png", dpi=200)

mean_ax = summary.pivot(index="round", columns="strategy", values="mean_fitness").plot(marker="o", figsize=(8, 4))
mean_ax.set_ylabel("Mean true fitness of recommended Top-k")
mean_ax.set_title("Mean recommendation fitness over 3 virtual rounds")
mean_fig = mean_ax.get_figure()
mean_fig.tight_layout()
mean_fig.savefig(ARTIFACTS / "virtual_evolution_mean_curve.png", dpi=200)

pos_ax = top_positions.pivot_table(
    index="position",
    columns="strategy",
    values="mutation_frequency",
    aggfunc="max",
).fillna(0).plot(kind="bar", figsize=(9, 4))
pos_ax.set_ylabel("Max mutation frequency in a Top-k round")
pos_ax.set_title("Recommended mutations concentrate on key positions")
pos_fig = pos_ax.get_figure()
pos_fig.tight_layout()
pos_fig.savefig(ARTIFACTS / "virtual_evolution_key_positions.png", dpi=200)
